# Semi-gradient SARSA with Tile Coding for MountainCar

This notebook applies **on-policy semi-gradient SARSA** to the continuous-state `MountainCar-v0` task.

The action-value approximation is

\[
\hat Q(s,a;\mathbf w)=\mathbf w_a^T\phi(s),
\]

where \(\phi(s)\) contains one active binary feature from each shifted tiling. Unlike Q-learning, SARSA bootstraps from the **next action selected by its epsilon-greedy behavior policy**.

## 1. Imports

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap

## 2. Environment and hyperparameters

All experimental parameters are grouped here for easy modification.

In [ ]:
# ============================================================
# PARAMETERS
# ============================================================

num_episodes = 25_000
gamma = 0.99

# Overall step-size parameter. Each transition activates num_tilings
# features, so the actual per-weight step size is alpha / num_tilings.
alpha = 0.5

epsilon_start = 1.0
epsilon_min = 0.01
epsilon_decay = 0.00025

num_tilings = 8
tiles_per_dim = 8

rolling_window = 500
diagnostic_window = 1_000
num_test_episodes = 1_000
policy_grid_size = 150

seed = 42
rng = np.random.default_rng(seed)

# ============================================================
# ENVIRONMENT
# ============================================================

env = gym.make("MountainCar-v0")
state_low = env.observation_space.low.astype(float)
state_high = env.observation_space.high.astype(float)
num_actions = env.action_space.n

print("State low:", state_low)
print("State high:", state_high)
print("Number of actions:", num_actions)

## 3. Tile coding

Each tiling covers position and velocity with an \(8\times8\) grid by default. Small, distinct offsets shift the tile boundaries, giving every state exactly one active feature per tiling.

In [ ]:
# ============================================================
# TILE CODING
# ============================================================

num_features_per_tiling = tiles_per_dim * tiles_per_dim
num_features = num_tilings * num_features_per_tiling

tile_width = (state_high - state_low) / tiles_per_dim


def get_active_features(state):
    """Return one valid active feature index from each tiling."""
    state = np.asarray(state, dtype=float)
    active_features = np.empty(num_tilings, dtype=np.int64)

    for tiling in range(num_tilings):
        # Use different offset patterns for position and velocity so that
        # tile boundaries do not line up across tilings.
        position_offset_fraction = tiling / num_tilings
        velocity_offset_fraction = ((2 * tiling + 1) % num_tilings) / num_tilings
        offset = np.array(
            [position_offset_fraction, velocity_offset_fraction]
        ) * tile_width

        tile_coordinates = np.floor(
            (state - state_low + offset) / tile_width
        ).astype(int)
        tile_coordinates = np.clip(
            tile_coordinates, 0, tiles_per_dim - 1
        )

        position_index, velocity_index = tile_coordinates
        local_index = position_index * tiles_per_dim + velocity_index
        active_features[tiling] = (
            tiling * num_features_per_tiling + local_index
        )

    return active_features


sample_state, _ = env.reset(seed=seed)
sample_features = get_active_features(sample_state)

print("Example state:", sample_state)
print("Active features:", sample_features)
print("Active feature count:", len(sample_features))
print("Total features:", num_features)

## 4. Linear action-value function

Each action owns a separate weight vector. Because the feature representation is sparse, evaluating \(Q(s,a)\) only requires summing the active weights.

In [ ]:
# ============================================================
# LINEAR Q-FUNCTION
# ============================================================

weights = np.zeros((num_actions, num_features), dtype=float)


def q_value(state, action):
    active = get_active_features(state)
    return float(np.sum(weights[action, active]))


def q_values(state):
    active = get_active_features(state)
    return np.sum(weights[:, active], axis=1)

## 5. Epsilon-greedy action selection

Greedy ties are broken randomly. During training, the same seeded NumPy generator controls exploration and tie-breaking.

In [ ]:
# ============================================================
# ACTION SELECTION AND SCHEDULES
# ============================================================

def choose_action(state, epsilon, random_generator=rng):
    """Choose from the epsilon-greedy behavior policy."""
    if random_generator.random() < epsilon:
        return int(random_generator.integers(num_actions))

    values = q_values(state)
    max_actions = np.flatnonzero(values == np.max(values))
    return int(random_generator.choice(max_actions))


def greedy_action(state, random_generator):
    """Pure exploitation with random tie-breaking."""
    values = q_values(state)
    max_actions = np.flatnonzero(values == np.max(values))
    return int(random_generator.choice(max_actions))


def epsilon_for_episode(episode):
    return epsilon_min + (epsilon_start - epsilon_min) * np.exp(
        -epsilon_decay * episode
    )


def rolling_mean(values, window):
    """Trailing mean that also produces values before a full window exists."""
    values = np.asarray(values, dtype=float)
    cumulative = np.cumsum(np.insert(values, 0, 0.0))
    result = np.empty_like(values)

    for end in range(1, len(values) + 1):
        start = max(0, end - window)
        result[end - 1] = (
            cumulative[end] - cumulative[start]
        ) / (end - start)

    return result

## 6. Semi-gradient SARSA training

For ordinary transitions, SARSA selects \(A_{t+1}\) once, uses it in

\[
R_{t+1}+\gamma\hat Q(S_{t+1},A_{t+1}),
\]

and carries that same action into the next loop iteration.

A true goal termination has no bootstrap term. A Gymnasium time-limit truncation still bootstraps because it is not a terminal state of the underlying MDP; the episode then ends because the wrapper requires a reset.

In [ ]:
# ============================================================
# SEMI-GRADIENT SARSA
# ============================================================

# Reset here so rerunning this cell always starts a fresh experiment.
weights.fill(0.0)
rng = np.random.default_rng(seed)

episode_lengths = np.zeros(num_episodes, dtype=np.int16)
episode_successes = np.zeros(num_episodes, dtype=np.int8)
episode_epsilons = np.zeros(num_episodes, dtype=float)
episode_mean_abs_td_error = np.zeros(num_episodes, dtype=float)

effective_alpha = alpha / num_tilings
training_stopped_early = False
episodes_completed = 0

# Fixed representative states used only for numerical diagnostics.
diagnostic_positions = np.linspace(state_low[0], state_high[0], 15)
diagnostic_velocities = np.linspace(state_low[1], state_high[1], 15)
diagnostic_states = np.array(
    [(position, velocity)
     for position in diagnostic_positions
     for velocity in diagnostic_velocities]
)

for episode in range(num_episodes):
    state, _ = env.reset(seed=seed + episode)
    epsilon = epsilon_for_episode(episode)
    action = choose_action(state, epsilon, rng)

    terminated = False
    truncated = False
    steps = 0
    episode_abs_td_errors = []

    while not (terminated or truncated):
        active = get_active_features(state)
        prediction = float(np.sum(weights[action, active]))

        next_state, reward, terminated, truncated, _ = env.step(action)

        if terminated:
            # Reaching the goal is a true terminal transition.
            next_action = None
            target = reward
        else:
            # This includes time-limit truncation: it is not an MDP terminal
            # state, so compute the SARSA bootstrap before ending the episode.
            next_action = choose_action(next_state, epsilon, rng)
            target = reward + gamma * q_value(next_state, next_action)

        td_error = target - prediction

        if not np.isfinite(td_error):
            print(
                f"WARNING: non-finite TD error at episode {episode + 1}, "
                f"step {steps + 1}. Training stopped."
            )
            training_stopped_early = True
            break

        weights[action, active] += effective_alpha * td_error

        if not np.all(np.isfinite(weights[action, active])):
            print(
                f"WARNING: non-finite weight at episode {episode + 1}, "
                f"step {steps + 1}. Training stopped."
            )
            training_stopped_early = True
            break

        episode_abs_td_errors.append(abs(td_error))
        steps += 1

        if not (terminated or truncated):
            # This is the defining SARSA ordering: next_action is not sampled
            # again. It becomes the action executed on the next iteration.
            state = next_state
            action = next_action

    episode_lengths[episode] = steps
    episode_successes[episode] = int(terminated)
    episode_epsilons[episode] = epsilon
    episode_mean_abs_td_error[episode] = (
        np.mean(episode_abs_td_errors) if episode_abs_td_errors else np.nan
    )
    episodes_completed = episode + 1

    if training_stopped_early:
        break

    if (episode + 1) % diagnostic_window == 0:
        recent_start = max(0, episode + 1 - diagnostic_window)
        recent_success = np.mean(
            episode_successes[recent_start:episode + 1]
        )
        recent_length = np.mean(
            episode_lengths[recent_start:episode + 1]
        )
        recent_td_error = np.nanmean(
            episode_mean_abs_td_error[recent_start:episode + 1]
        )
        representative_q_values = np.array(
            [q_values(sample) for sample in diagnostic_states]
        )
        max_predicted_q = np.max(np.abs(representative_q_values))

        print(
            f"Episode {episode + 1:5d} | "
            f"epsilon = {epsilon:.4f} | "
            f"recent success = {recent_success:.3f} | "
            f"recent length = {recent_length:.1f} | "
            f"mean |TD| = {recent_td_error:.3f} | "
            f"max |w| = {np.max(np.abs(weights)):.3f} | "
            f"mean |w| = {np.mean(np.abs(weights)):.3f} | "
            f"sample max |Q| = {max_predicted_q:.3f}"
        )

        if not (
            np.all(np.isfinite(representative_q_values))
            and np.all(np.isfinite(weights))
        ):
            print(
                "WARNING: NaN or infinity detected in weights or predicted "
                "Q-values. Training stopped."
            )
            training_stopped_early = True
            break

env.close()

# Trim statistics if a numerical problem stopped training early.
episode_lengths = episode_lengths[:episodes_completed]
episode_successes = episode_successes[:episodes_completed]
episode_epsilons = episode_epsilons[:episodes_completed]
episode_mean_abs_td_error = episode_mean_abs_td_error[:episodes_completed]

print("\nTraining complete.")
print("Episodes completed:", episodes_completed)
print("Maximum absolute weight:", np.max(np.abs(weights)))
print("Mean absolute weight:", np.mean(np.abs(weights)))

## 7. Training diagnostics

In [ ]:
print(
    "Final rolling training success:",
    np.mean(episode_successes[-min(rolling_window, episodes_completed):])
)
print(
    "Final rolling episode length:",
    np.mean(episode_lengths[-min(rolling_window, episodes_completed):])
)
print(
    "Final recent mean absolute TD error:",
    np.nanmean(
        episode_mean_abs_td_error[
            -min(diagnostic_window, episodes_completed):
        ]
    )
)
print("Final epsilon:", episode_epsilons[-1])

## 8. Learning curves

In [ ]:
episode_axis = np.arange(1, episodes_completed + 1)
rolling_success = rolling_mean(episode_successes, rolling_window)
rolling_length = rolling_mean(episode_lengths, rolling_window)
rolling_abs_td_error = rolling_mean(
    np.nan_to_num(episode_mean_abs_td_error, nan=0.0),
    rolling_window
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(episode_axis, rolling_success)
axes[0].set(
    xlabel="Episode",
    ylabel="Rolling success rate",
    title=f"Success over previous {rolling_window} episodes",
    ylim=(0, 1.02),
)
axes[0].grid(alpha=0.3)

axes[1].plot(episode_axis, rolling_length, color="tab:orange")
axes[1].set(
    xlabel="Episode",
    ylabel="Rolling average episode length",
    title=f"Length over previous {rolling_window} episodes",
)
axes[1].grid(alpha=0.3)

axes[2].plot(episode_axis, rolling_abs_td_error, color="tab:green")
axes[2].set(
    xlabel="Episode",
    ylabel="Rolling mean absolute TD error",
    title=f"|TD error| over previous {rolling_window} episodes",
)
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Independent greedy evaluation

Evaluation uses pure exploitation (epsilon = 0) and fresh reset seeds. Randomness is used only to break exact ties.

In [ ]:
def evaluate_greedy_policy(test_episodes=num_test_episodes):
    test_env = gym.make("MountainCar-v0")
    test_rng = np.random.default_rng(seed + 1_000_000)

    test_successes = np.zeros(test_episodes, dtype=np.int8)
    test_episode_lengths = np.zeros(test_episodes, dtype=np.int16)

    for episode in range(test_episodes):
        state, _ = test_env.reset(seed=seed + 1_000_000 + episode)
        terminated = False
        truncated = False

        while not (terminated or truncated):
            action = greedy_action(state, test_rng)
            state, _, terminated, truncated, _ = test_env.step(action)
            test_episode_lengths[episode] += 1

        test_successes[episode] = int(terminated)

    test_env.close()
    return test_successes, test_episode_lengths


test_successes, test_episode_lengths = evaluate_greedy_policy()

print(f"Test successes: {np.sum(test_successes)} / {num_test_episodes}")
print(f"Test success rate: {np.mean(test_successes):.3f}")
print(f"Average test episode length: {np.mean(test_episode_lengths):.2f}")
print(f"Minimum test episode length: {np.min(test_episode_lengths)}")
print(f"Maximum test episode length: {np.max(test_episode_lengths)}")

## 10. Learned greedy policy

The continuous position-velocity plane is sampled on a fine grid. Each grid point is colored by its maximizing action.

In [ ]:
positions = np.linspace(
    state_low[0], state_high[0], policy_grid_size
)
velocities = np.linspace(
    state_low[1], state_high[1], policy_grid_size
)

policy = np.empty(
    (policy_grid_size, policy_grid_size), dtype=np.int8
)

for velocity_index, velocity in enumerate(velocities):
    for position_index, position in enumerate(positions):
        # argmax gives a stable map; evaluation episodes above retain random
        # tie-breaking when actions are actually selected.
        policy[velocity_index, position_index] = np.argmax(
            q_values(np.array([position, velocity]))
        )

cmap = ListedColormap(["tab:blue", "tab:gray", "tab:orange"])
norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5], cmap.N)

plt.figure(figsize=(10, 6))
image = plt.imshow(
    policy,
    origin="lower",
    aspect="auto",
    extent=[
        state_low[0],
        state_high[0],
        state_low[1],
        state_high[1],
    ],
    cmap=cmap,
    norm=norm,
)
colorbar = plt.colorbar(image, ticks=[0, 1, 2])
colorbar.ax.set_yticklabels(["Left", "Coast", "Right"])

plt.axvline(
    x=0.5, linestyle="--", color="purple", label="Goal position"
)
plt.xlabel("Position")
plt.ylabel("Velocity")
plt.title("Greedy SARSA policy with tile-coded linear approximation")
plt.legend()
plt.tight_layout()
plt.show()

## 11. Optional rendered greedy episode

Run the following cell only when a graphical display is available. It opens a Gymnasium rendering window and closes it after one episode.

In [ ]:
# Set this to True when you explicitly want to watch one trajectory.
# The guard prevents a rendering window from opening during Run All.
RENDER_EPISODE = False

if RENDER_EPISODE:
    render_env = gym.make("MountainCar-v0", render_mode="human")
    render_rng = np.random.default_rng(seed + 2_000_000)

    state, _ = render_env.reset(seed=seed + 2_000_000)
    terminated = False
    truncated = False
    render_steps = 0

    while not (terminated or truncated):
        action = greedy_action(state, render_rng)
        state, _, terminated, truncated, _ = render_env.step(action)
        render_steps += 1

    render_env.close()
    print("Rendered episode success:", terminated)
    print("Rendered episode length:", render_steps)
else:
    print("Set RENDER_EPISODE = True to render a greedy episode.")

## Saving the trained SARSA model

This cell runs after training and saves the learned weights together with the metadata needed to reconstruct the tile-coded function approximation. Because this notebook uses tile coding rather than radial basis functions, `centers` stores the two-dimensional offset of every tiling and `sigma` stores the tile width in each state dimension.

In [ ]:
from pathlib import Path

model_path = Path("models") / "sarsa_mountaincar_model.npz"
model_path.parent.mkdir(parents=True, exist_ok=True)

# Native tile-coding geometry saved under the requested compatibility keys.
centers = np.array([
    [
        tiling / num_tilings,
        ((2 * tiling + 1) % num_tilings) / num_tilings,
    ]
    for tiling in range(num_tilings)
]) * tile_width
sigma = tile_width.copy()

np.savez_compressed(
    model_path,
    weights=weights,
    centers=centers,
    sigma=sigma,
    gamma=gamma,
    alpha=alpha,
    environment="MountainCar-v0",
    episodes=episodes_completed,
    epsilon_initial=epsilon_start,
    epsilon_final=episode_epsilons[-1],
    state_low=state_low,
    state_high=state_high,
    num_tilings=num_tilings,
    tiles_per_dim=tiles_per_dim,
)

print(f"Saved trained SARSA model to: {model_path}")

## Loading the trained SARSA model without retraining

The archive is portable and can be loaded from a different notebook or Python script. For this tile-coded model, `sigma` is a two-element tile-width array rather than a scalar.

In [ ]:
with np.load("models/sarsa_mountaincar_model.npz") as model:
    weights = model["weights"].copy()
    centers = model["centers"].copy()
    sigma = model["sigma"].copy()
    gamma = float(model["gamma"])
    alpha = float(model["alpha"])
    environment = str(model["environment"])
    episodes = int(model["episodes"])
    epsilon_initial = float(model["epsilon_initial"])
    epsilon_final = float(model["epsilon_final"])

    # Additional native metadata needed to rebuild get_active_features.
    state_low = model["state_low"].copy()
    state_high = model["state_high"].copy()
    num_tilings = int(model["num_tilings"])
    tiles_per_dim = int(model["tiles_per_dim"])

print("Loaded SARSA model")
print("Environment:", environment)
print("Training episodes:", episodes)
print("Gamma:", gamma)
print("Alpha:", alpha)
print("Initial epsilon:", epsilon_initial)
print("Final epsilon:", epsilon_final)